# Filter AMR papers with LLM

Use the `llm.py` helper to ask a model whether a paper's abstract asserts that a pathogen is showing **significant resistance** to a given antibiotic.


In [ ]:
#!pip install openai requests pandas lxml

In [ ]:

import pandas as pd

from llm import create_llm_client
from amr.filtering import (
    DEFAULT_SYSTEM_PROMPT,
    prepare_pmids_to_review,
    run_llm_filter,
)

# --- Explicit configuration ---
SYSTEM_PROMPT = """
You are an expert microbiologist reviewing evidence. Given a title and abstract, decide whether the authors explicitly state or conclude that the specified pathogen is showing clinically meaningful resistance to the specified antibiotic.

Rules:
- Answer "yes" only if the abstract clearly reports resistance, reduced susceptibility, treatment failure due to resistance, or emergence of resistant isolates for that pathogen/antibiotic pair.
- Answer "no" if the abstract does not make that claim, is about a different organism/drug, only mentions testing without resistance, or only provides background.
- Answer "uncertain" if the abstract is missing, ambiguous, or only indirectly related.
- Base the decision only on the provided text; do not infer beyond it.

Return compact JSON with keys: decision (yes/no/uncertain) and rationale (one short sentence).
"""

LLM_PROVIDER = "openai"
MODEL_NAME = "gpt-4o-mini"
NCBI_EMAIL = "your_email@example.com"
MAX_PMIDS = 200  # safety cap per run
RATE_LIMIT = 0.35  # seconds between PubMed calls
OUTPUT_PATH = "pmids_results_filtered.parquet"
LOG_PATH = "pmid_filter.log"
MAX_LLM_CALLS = None  # set to an integer to cap spend per run
DRY_RUN = False  # set True to skip LLM calls
SAVE_EVERY = 50  # checkpoint frequency

llm_kwargs = {}
if MODEL_NAME:
    llm_kwargs["model"] = MODEL_NAME

llm = create_llm_client(provider=LLM_PROVIDER, **llm_kwargs)
print(f"Using provider={LLM_PROVIDER}, model={MODEL_NAME or 'provider default'}; dry_run={DRY_RUN}; max_llm_calls={MAX_LLM_CALLS}")


In [ ]:

# Load already-reviewed decisions if present
if os.path.exists(OUTPUT_PATH):
    reviewed_df = pd.read_parquet(OUTPUT_PATH)
    reviewed = set(reviewed_df['pmid'].astype(str)) if not reviewed_df.empty else set()
    print(f"Resuming with {len(reviewed)} PMIDs already reviewed from {OUTPUT_PATH}")
else:
    reviewed_df = None
    reviewed = set()


In [ ]:

pmid_df = pd.read_parquet("pmids_results.parquet")
long_df = prepare_pmids_to_review(pmid_df, max_pmids=MAX_PMIDS)
# Drop already processed PMIDs if resuming
if reviewed:
    long_df = long_df[~long_df['pmid'].astype(str).isin(reviewed)].reset_index(drop=True)
print(f"PMIDs remaining to review this run: {len(long_df)}")
long_df.head()


In [ ]:

filtered_df = run_llm_filter(
    long_df,
    llm,
    email=NCBI_EMAIL,
    rate_limit=RATE_LIMIT,
    save_every=SAVE_EVERY,
    system_prompt=SYSTEM_PROMPT,
    progress_every=10,
    resume_from=reviewed_df,
    seen_pmids=reviewed,
    output_path=OUTPUT_PATH,
    log_path=LOG_PATH,
    max_llm_calls=MAX_LLM_CALLS,
    dry_run=DRY_RUN,
)
filtered_df.head()


In [ ]:

print("Summary (this run):")
total_processed = len(filtered_df)
positives = (filtered_df['decision'].str.lower() == 'yes').sum()
fraction = positives / total_processed if total_processed else 0
print(f"Processed {total_processed} abstracts; positives {positives} ({fraction:.3f}).")

if os.path.exists(OUTPUT_PATH):
    all_df = pd.read_parquet(OUTPUT_PATH)
    all_total = len(all_df)
    all_pos = (all_df['decision'].str.lower() == 'yes').sum()
    all_fraction = all_pos / all_total if all_total else 0
    print(f"Cumulative: {all_total} abstracts; positives {all_pos} ({all_fraction:.3f}).")

if os.path.exists(LOG_PATH):
    tail = open(LOG_PATH, 'r', encoding='utf-8').read().splitlines()
    tail = tail[-10:] if tail else []
    print("
Log tail:")
    for line in tail:
        print(line)


In [ ]:

print(f"Saved results to {OUTPUT_PATH}")
filtered_df['decision'].value_counts()


In [ ]:

# Build positive-only dataset and save
pmid_df_full = pd.read_parquet("pmids_results.parquet")

# Merge decisions onto exploded PMIDs
positive_flags = filtered_df[filtered_df["decision"].str.lower() == "yes"][["pmid"]].copy()
positive_rows = (
    pmid_df_full.explode("pmids")
    .assign(pmid=lambda d: d["pmids"].astype(str).str.extract(r"(\d+)")[0])
    .dropna(subset=["pmid"])
    .merge(positive_flags, on="pmid", how="inner")
)

# Reassemble to original grain
pmids_positive = (
    positive_rows.groupby(["pathogen", "antibiotic", "start_date", "end_date"], as_index=False)
    .agg(
        pmids=("pmid", list),
        count=("pmid", "size"),
        venue_histogram=("venue_histogram", "first"),
    )
)

positive_path = "pmids_results_positive.parquet"
pmids_positive.to_parquet(positive_path, index=False)
print(f"Saved positive-only dataset to {positive_path} with {len(pmids_positive)} rows")
pmids_positive.head()
